# 07 -- Create Staff Subscription Orders

Staff-migration variant of `07_Create_Subscription_Orders.ipynb`. All the
field-mapping / plan-resolution / order-building logic is shared (lives in
`onebill_common.py`, same functions the Williams pipeline uses) -- this
notebook just loads the staff pipeline's data and calls them.

For every subscription that got an address in `06_Create_Staff_Addresses.ipynb`:

1. Resolve its plan -- `03_Match_Plan_Codes.ipynb` mapping (shared, unchanged
   for staff), falling back to `STATIC_FALLBACK_PLAN` if unmatched.
2. Optionally attach a primary-contact summary from `01_Fetch_Contacts.ipynb`
   (see `ATTACH_CONTACT_SUMMARY_TO_ORDER`) -- mostly redundant here since
   every staff account already has its real contacts attached at account
   creation (unlike the Williams bucket accounts, where this was the only
   place a customer's contact info showed up), but harmless to leave on.
3. Build the order payload per the same field mapping as the Williams
   pipeline.
4. `POST /rest/OrderService/v1/order`.

**vBill -> OneBill field mapping** -- identical to `07_Create_Subscription_Orders.ipynb`; see that notebook's intro for the full table.

## 1. Setup -- load everything the previous notebooks produced

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from onebill_common import *  # noqa: F401,F403
from concurrent.futures import ThreadPoolExecutor, as_completed

logger = get_logger("create_staff_subscription_orders")

df_subscriptions   = load_staff_subscriptions_resolved()
df_address_results = load_df("staff_address_results", dtype={"ship_add_id": str})
df_plan_mapping     = try_load_df("plan_mapping", dtype=str)
df_contacts          = try_load_df("contacts")

if df_plan_mapping is None:
    logger.warning("03_plan_code_mapping.csv not found -- every subscription falls back to STATIC_FALLBACK_PLAN.")
    df_plan_mapping = pd.DataFrame(columns=["PlanCode", "product_name", "priceplan_name"])

if df_contacts is None and ATTACH_CONTACT_SUMMARY_TO_ORDER:
    logger.warning("01_contacts_by_account.csv not found -- orders will be created without a contact summary.")

logger.info(
    f"{len(df_subscriptions):,} subscriptions, {len(df_address_results):,} address results, "
    f"{len(df_plan_mapping):,} plan mappings, "
    f"{len(df_contacts) if df_contacts is not None else 0:,} contact summaries"
)


In [ ]:
# HARDCODED_TOKEN = "c5bf2481-f3fd-498b-a260-27c6e783782f"

# token_manager._token = HARDCODED_TOKEN
# token_manager._expires_at = datetime.now() + timedelta(hours=1)  # adjust to match the real token's actual TTL

## 2. Join subscriptions with their address result

In [ ]:
df_work = df_subscriptions.merge(
    df_address_results[["SubscriptionUSN", "status", "ship_add_id", "error"]].rename(
        columns={"status": "AddressStatus", "error": "AddressError"}
    ),
    on="SubscriptionUSN", how="left",
)

ready = df_work[df_work["AddressStatus"].isin(["created", "exists"])]
not_ready = df_work[~df_work["AddressStatus"].isin(["created", "exists"])]
logger.info(f"{len(ready):,} subscriptions have an address and are ready for order creation; "
            f"{len(not_ready):,} do not and will be skipped")
not_ready[["SubscriptionUSN", "AddressStatus", "AddressError"]].head(20)


## 3. Plan resolution

In [ ]:
resolve_plan = make_plan_resolver(df_plan_mapping)


## 4. Contact-summary lookup (optional)

Joins each subscription's `AccountCode` against `01_Fetch_Contacts.ipynb`'s output -- for staff this is the subscription's own real account, not a shared bucket, so this is a secondary confirmation rather than the primary way contact info reaches the account.

In [ ]:
contact_summary_attributes = make_contact_summary_fn(df_contacts)


## 5. Build the order payload (field mapping)

In [ ]:
# build_subscription_order_payload is defined in onebill_common.py (shared
# with every other pipeline) -- nothing to define here.


## 6. Per-subscription worker

In [ ]:
# create_order_for_subscription is defined in onebill_common.py (shared with
# every other pipeline) -- nothing to define here.


## 7. Run (parallel driver)

In [ ]:
order_results_df = create_all_orders(ready, resolve_plan, contact_summary_attributes)
order_results_df.head(20)


## 8. Save

In [ ]:
save_df("staff_order_results", order_results_df)
